In [13]:
#! pip install wikipedia
#! pip install -U ddgs
#! pip install langchain

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.3/125.3 kB 546.1 kB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.4/245.4 kB 821.4 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.5/160.5 kB 834.1 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.5/182.5 kB 772.3 kB/s eta 0:00:00a 0:00:01
  Attempting uninstall: websockets
    Found existing installation: websockets 16.0
    Uninstalling websockets-16.0:
      Successfully uninstalled websockets-16.0
  Attempting uninstall: langgraph-sdk
    Found existing installation: langgraph-sdk 0.3.14
    Uninstalling langgraph-sdk-0.3.14:
      Successfully uninstalled langgraph-sdk-0.3.14
  Attempting uninstall: langgraph
    Found existing installation: langgraph 1.2.1
    Uninstalling langgraph-1.2.1:
      Successfully uninstalled langgraph-1.2.1


In [33]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
# import langchain tool
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_community.utilities import DuckDuckGoSearchAPIWrapper
from langchain.tools import tool

load_dotenv()
#==========setup .env ===========
if os.getenv("GEMINI_API_KEY") is None:
    raise ValueError("GEMINI_API_KEY environment variable not set. Please set it in your .env file.")
else:
    print("GEMINI_API_KEY environment variable is set.")

GEMINI_API_KEY environment variable is set.


In [34]:
#==========setup model ===========
llm=ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite",
                                max_tokens=2048,temperature=0.5,
                                 api_key=os.getenv("GEMINI_API_KEY"))

In [31]:
ddg_api_wrapper = DuckDuckGoSearchAPIWrapper(
    max_results=2,
    source="text")

@tool("ddg-search")
def ddg_search(query: str) -> str:
    """a tool to search on duckduckgo and return the result"""
    ddg=DuckDuckGoSearchRun(api_wrapper=ddg_api_wrapper)
    return ddg.invoke(query)

#==========test the tool ===========
test_query="What is the capital of France?"
response=ddg_search.invoke({"query": test_query})
print(response)
    

With 200,000 inhabitants in 1328, Paris, then already the capital of France, was the most populous city of Europe. By comparison, London in 1300 had 80,000 inhabitants.[31] Around 1340 the city had between 250,000 and 300,000 inhabitants.[32] By the early fourteenth century, so much... Paris is the capital and most populous city of France.


In [44]:
@tool("personal-info")
def personal_info(name: str) -> str:
    """a tool to return personal info about user name
    arguments:
    name: the name of user
    return: personal info about user name
    """
    return f"my name is {name} and i am a AI engineer 555555"

In [45]:
personal_info_response=personal_info.invoke({"name": "over-mind"})
print(personal_info_response)

my name is over-mind and i am a AI engineer 555555


In [49]:
tool_stack=[ddg_search,personal_info]
llm_with_tools=llm.bind_tools(tool_stack)
response=llm_with_tools.invoke(" what is personal info of over-mind?")
response

AIMessage(content=[], additional_kwargs={'function_call': {'name': 'personal-info', 'arguments': '{"name": "over-mind"}'}, '__gemini_function_call_thought_signatures__': {'ffe81394-ec88-49da-a036-e17d26b26e04': 'EjQKMgEMOdbHtIoVVcka7wAOU2v1ReeN9dLIgcb6IuhFCN7WaW017Uws9LUJDkDB3k89Yiw2'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e8e2a-a187-7c52-9e94-5cc110317ecc-0', tool_calls=[{'name': 'personal-info', 'args': {'name': 'over-mind'}, 'id': 'ffe81394-ec88-49da-a036-e17d26b26e04', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 127, 'output_tokens': 18, 'total_tokens': 145, 'input_token_details': {'cache_read': 0}})

In [50]:
response.tool_calls

[{'name': 'personal-info',
  'args': {'name': 'over-mind'},
  'id': 'ffe81394-ec88-49da-a036-e17d26b26e04',
  'type': 'tool_call'}]

In [52]:

response=llm_with_tools.invoke(" who is president of egypt now?")
response.tool_calls

[{'name': 'ddg-search',
  'args': {'query': 'current president of Egypt'},
  'id': '5a3f3ffb-8f7f-41de-98da-9063a4ab2552',
  'type': 'tool_call'}]